# Comparative Study of CNN Models on the Kvasir Capsule Dataset with Imbalance Handling Techniques

**Course:** Deep Learning Lab — Minor Project  
**Dataset:** Kvasir Capsule  
**Platform:** Kaggle (GPU enabled)  

---

This notebook compares three CNN architectures (EfficientNet-B4, ResNet101V2, InceptionV3) trained on three dataset variants:
- Original imbalanced dataset
- Undersampled dataset
- Undersampled + augmented (balanced) dataset with Mixup training

Additional techniques: Feature-Space SMOTE, Test-Time Augmentation (TTA), GradCAM, ROC-AUC analysis.

## Imports and Environment Setup

In [ ]:
# install additional dependencies not present in kaggle by default
import subprocess
subprocess.run(["pip", "install", "imbalanced-learn", "opencv-python-headless", "-q"], check=True)

In [ ]:
import os
import gc
import shutil
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import cv2

from pathlib import Path
from collections import Counter, defaultdict
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, backend as K
from tensorflow.keras.applications import EfficientNetB4, ResNet101V2, InceptionV3
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_fscore_support
)
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE
from imblearn.over_sampling import SMOTE

warnings.filterwarnings("ignore")

# fix random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# paths
DATA_ROOT = Path("/kaggle/input/datasets/harshvardhansg/kavasir-capsule")
WORK_DIR  = Path("/kaggle/working")

# dataset variant save paths
DS1_PATH = WORK_DIR / "dataset1_imbalanced"
DS2_PATH = WORK_DIR / "dataset2_undersampled"
DS3_PATH = WORK_DIR / "dataset3_augmented"

IMG_SIZE        = 224
BATCH_SIZE      = 32
EPOCHS          = 20
UNDERSAMPLE_CAP = 600  # max images per class for undersampling

# valid image file extensions
VALID_EXTS = {".jpg", ".jpeg", ".png"}

print("tensorflow version:", tf.__version__)
print("gpu available:", tf.config.list_physical_devices('GPU'))

---
## Section 1: Dataset Exploration

In [ ]:
# discover class folders — sorted for deterministic ordering
class_dirs  = sorted([d for d in DATA_ROOT.iterdir() if d.is_dir()])
CLASS_NAMES = [d.name for d in class_dirs]
NUM_CLASSES = len(CLASS_NAMES)

print(f"total classes found: {NUM_CLASSES}")
print("class names:", CLASS_NAMES)

In [ ]:
def count_images(folder: Path) -> int:
    """
    count valid image files inside a class folder, searching recursively.
    the kvasir capsule dataset stores images in subdirectories within each
    class folder, so rglob is required instead of a flat iterdir scan.
    """
    return sum(
        1 for f in folder.rglob("*")
        if f.is_file() and f.suffix.lower() in VALID_EXTS
    )

class_counts = {cls: count_images(DATA_ROOT / cls) for cls in CLASS_NAMES}

# sort descending for readability
class_counts_sorted = dict(sorted(class_counts.items(), key=lambda x: x[1], reverse=True))

print("\nimage count per class:")
for cls, cnt in class_counts_sorted.items():
    print(f"  {cls:<35} {cnt}")

In [ ]:
# bar chart of class distribution
names     = list(class_counts_sorted.keys())
counts    = list(class_counts_sorted.values())
max_count = max(counts)
min_count = min(counts)

colors = [
    "#e74c3c" if c == max_count else ("#2ecc71" if c == min_count else "#3498db")
    for c in counts
]

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.barh(names, counts, color=colors, edgecolor="white", linewidth=0.5)

# annotate bar values
for bar, cnt in zip(bars, counts):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height() / 2,
            str(cnt), va="center", fontsize=9)

ax.set_xlabel("Number of Images", fontsize=11)
ax.set_title("Class Distribution in Kvasir Capsule Dataset", fontsize=13, fontweight="bold")

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#e74c3c", label="majority class"),
    Patch(facecolor="#2ecc71", label="minority class"),
    Patch(facecolor="#3498db", label="other classes"),
]
ax.legend(handles=legend_elements, loc="lower right")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
def get_one_sample(class_path: Path) -> np.ndarray:
    """
    return the first valid image found recursively inside a class folder as an rgb array.
    uses rglob to handle datasets where images are nested in subdirectories.
    """
    for f in sorted(class_path.rglob("*")):
        if f.is_file() and f.suffix.lower() in VALID_EXTS:
            img = cv2.imread(str(f))
            if img is not None:
                return cv2.cvtColor(cv2.resize(img, (IMG_SIZE, IMG_SIZE)), cv2.COLOR_BGR2RGB)
    return None

cols = 4
rows = (NUM_CLASSES + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.5, rows * 3.5))
axes = axes.flatten()

for i, cls in enumerate(CLASS_NAMES):
    img = get_one_sample(DATA_ROOT / cls)
    if img is not None:
        axes[i].imshow(img)
    axes[i].set_title(cls, fontsize=8, wrap=True)
    axes[i].axis("off")

# hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.suptitle("Sample Image Per Class — Kvasir Capsule", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## Section 2: Data Preprocessing and Stratified Split

In [ ]:
# build a flat list of (image_path, class_index) for stratified splitting.
# rglob is used here for the same reason as above: images may be nested
# one level deeper than the class folder (e.g. class/subdir/image.jpg).
all_paths  = []
all_labels = []

for idx, cls in enumerate(CLASS_NAMES):
    cls_dir = DATA_ROOT / cls
    for f in sorted(cls_dir.rglob("*")):
        if f.is_file() and f.suffix.lower() in VALID_EXTS:
            all_paths.append(str(f))
            all_labels.append(idx)

all_paths  = np.array(all_paths)
all_labels = np.array(all_labels)

print(f"total images found: {len(all_paths)}")

# first split: 85% train+val | 15% test
paths_trainval, paths_test, labels_trainval, labels_test = train_test_split(
    all_paths, all_labels,
    test_size=0.15,
    stratify=all_labels,
    random_state=SEED
)

# second split: 70% train | 15% val (from the 85% pool)
val_ratio_adjusted = 0.15 / 0.85  # approx 17.6% of the pool = 15% of total
paths_train, paths_val, labels_train, labels_val = train_test_split(
    paths_trainval, labels_trainval,
    test_size=val_ratio_adjusted,
    stratify=labels_trainval,
    random_state=SEED
)

print(f"train : {len(paths_train)} images")
print(f"val   : {len(paths_val)} images")
print(f"test  : {len(paths_test)} images")

---
## Section 3: Three Dataset Variants Saved to Disk

In [ ]:
def copy_split_to_disk(paths: np.ndarray, labels: np.ndarray, dest_root: Path, split: str):
    """
    copy image files into a class-folder structure at dest_root/split/class_name/.
    files are copied directly from source — images are never loaded into ram.
    """
    for path, label in zip(paths, labels):
        cls_name = CLASS_NAMES[label]
        dest_dir = dest_root / split / cls_name
        dest_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, dest_dir / Path(path).name)


# --- dataset 1: original imbalanced ---
print("saving dataset 1 (imbalanced) to disk...")
if not DS1_PATH.exists():
    copy_split_to_disk(paths_train, labels_train, DS1_PATH, "train")
    copy_split_to_disk(paths_val,   labels_val,   DS1_PATH, "val")
    copy_split_to_disk(paths_test,  labels_test,  DS1_PATH, "test")
    print("  dataset 1 saved.")
else:
    print("  dataset 1 already exists, skipping.")

In [ ]:
# --- dataset 2: undersampled ---
# for any class with more than UNDERSAMPLE_CAP training images, randomly keep only that many
print("saving dataset 2 (undersampled) to disk...")

if not DS2_PATH.exists():
    train_by_class = defaultdict(list)
    for p, l in zip(paths_train, labels_train):
        train_by_class[l].append(p)

    us_paths_train, us_labels_train = [], []
    for cls_idx, cls_paths in train_by_class.items():
        if len(cls_paths) > UNDERSAMPLE_CAP:
            selected = random.sample(cls_paths, UNDERSAMPLE_CAP)
        else:
            selected = cls_paths
        us_paths_train.extend(selected)
        us_labels_train.extend([cls_idx] * len(selected))

    us_paths_train  = np.array(us_paths_train)
    us_labels_train = np.array(us_labels_train)

    copy_split_to_disk(us_paths_train, us_labels_train, DS2_PATH, "train")
    copy_split_to_disk(paths_val,      labels_val,      DS2_PATH, "val")
    copy_split_to_disk(paths_test,     labels_test,     DS2_PATH, "test")
    print("  dataset 2 saved.")
else:
    # reload paths in case this cell runs on a resumed kernel
    us_paths_train = np.array([
        str(f)
        for cls in CLASS_NAMES
        for f in (DS2_PATH / "train" / cls).iterdir()
        if f.suffix.lower() in VALID_EXTS
    ])
    us_labels_train = np.array([
        CLASS_NAMES.index(Path(p).parent.name)
        for p in us_paths_train
    ])
    print("  dataset 2 already exists, paths reloaded.")

# show resulting class distribution
us_class_counts = Counter(us_labels_train)
print("\nundersampled train class counts:")
for i, cnt in sorted(us_class_counts.items()):
    print(f"  {CLASS_NAMES[i]:<35} {cnt}")

In [ ]:
# --- augmentation helpers used for dataset 3 ---

def apply_clahe(img_rgb: np.ndarray) -> np.ndarray:
    """apply clahe on the l-channel of lab colorspace and return rgb."""
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l_eq  = clahe.apply(l)
    return cv2.cvtColor(cv2.merge([l_eq, a, b]), cv2.COLOR_LAB2RGB)


def apply_gaussian_noise(img_rgb: np.ndarray, mean=0, sigma=15) -> np.ndarray:
    """add gaussian noise to an rgb uint8 image."""
    noise = np.random.normal(mean, sigma, img_rgb.shape).astype(np.int16)
    return np.clip(img_rgb.astype(np.int16) + noise, 0, 255).astype(np.uint8)


def augment_image(img_rgb: np.ndarray) -> np.ndarray:
    """
    apply a randomly chosen augmentation to produce one synthetic image.
    strategies: hflip, rotate, shift, zoom, clahe, gaussian noise.
    """
    choice = random.randint(0, 5)
    h, w   = img_rgb.shape[:2]

    if choice == 0:
        # horizontal flip
        return np.fliplr(img_rgb)

    elif choice == 1:
        # random rotation in [-20, 20] degrees
        angle = random.uniform(-20, 20)
        M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
        return cv2.warpAffine(img_rgb, M, (w, h), borderMode=cv2.BORDER_REFLECT)

    elif choice == 2:
        # width/height shift by up to 20%
        tx = random.uniform(-0.2, 0.2) * w
        ty = random.uniform(-0.2, 0.2) * h
        M  = np.float32([[1, 0, tx], [0, 1, ty]])
        return cv2.warpAffine(img_rgb, M, (w, h), borderMode=cv2.BORDER_REFLECT)

    elif choice == 3:
        # zoom in by cropping center and resizing back
        factor = random.uniform(1.0, 1.2)
        new_h, new_w = int(h / factor), int(w / factor)
        y1 = (h - new_h) // 2
        x1 = (w - new_w) // 2
        return cv2.resize(img_rgb[y1:y1 + new_h, x1:x1 + new_w], (w, h))

    elif choice == 4:
        return apply_clahe(img_rgb)

    else:
        return apply_gaussian_noise(img_rgb)

In [ ]:
# --- dataset 3: undersampled + augmented ---
# start from dataset 2; augment minority classes until all reach UNDERSAMPLE_CAP
print("saving dataset 3 (undersampled + augmented) to disk...")

if not DS3_PATH.exists():
    # val and test splits are copied as-is
    copy_split_to_disk(paths_val,  labels_val,  DS3_PATH, "val")
    copy_split_to_disk(paths_test, labels_test, DS3_PATH, "test")

    # build per-class list from the undersampled train paths
    train_by_class_us = defaultdict(list)
    for p, l in zip(us_paths_train, us_labels_train):
        train_by_class_us[l].append(p)

    for cls_idx, cls_paths in train_by_class_us.items():
        cls_name = CLASS_NAMES[cls_idx]
        dest_dir = DS3_PATH / "train" / cls_name
        dest_dir.mkdir(parents=True, exist_ok=True)

        # copy existing images from dataset 2
        for p in cls_paths:
            shutil.copy2(p, dest_dir / Path(p).name)

        # generate synthetic images for minority classes
        needed = UNDERSAMPLE_CAP - len(cls_paths)
        if needed > 0:
            aug_count  = 0
            src_cycle  = cls_paths.copy()
            random.shuffle(src_cycle)
            idx_cycle  = 0
            while aug_count < needed:
                src_path = src_cycle[idx_cycle % len(src_cycle)]
                raw = cv2.imread(src_path)
                if raw is None:
                    idx_cycle += 1
                    continue
                img_rgb  = cv2.cvtColor(cv2.resize(raw, (IMG_SIZE, IMG_SIZE)), cv2.COLOR_BGR2RGB)
                aug_img  = augment_image(img_rgb)
                save_name = f"aug_{cls_idx}_{aug_count:05d}.jpg"
                cv2.imwrite(str(dest_dir / save_name),
                            cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR))
                aug_count += 1
                idx_cycle += 1
            print(f"  {cls_name}: added {needed} augmented images")

    print("  dataset 3 saved.")
else:
    print("  dataset 3 already exists, skipping.")

### Section 3b: Augmentation Visualization

In [ ]:
# pick the minority class (fewest images in the undersampled train split)
minority_cls_idx  = min(us_class_counts, key=us_class_counts.get)
minority_cls_name = CLASS_NAMES[minority_cls_idx]
print(f"minority class used for augmentation demo: {minority_cls_name}")

# load one sample from the minority class
sample_paths = [p for p, l in zip(us_paths_train, us_labels_train) if l == minority_cls_idx]
sample_path  = random.choice(sample_paths)
raw          = cv2.imread(sample_path)
sample_img   = cv2.cvtColor(cv2.resize(raw, (IMG_SIZE, IMG_SIZE)), cv2.COLOR_BGR2RGB)

# generate deterministic augmentation variants for the demo
h, w = sample_img.shape[:2]

flipped = np.fliplr(sample_img)

M_rot   = cv2.getRotationMatrix2D((w / 2, h / 2), 20, 1.0)
rotated = cv2.warpAffine(sample_img, M_rot, (w, h), borderMode=cv2.BORDER_REFLECT)

tx, ty  = int(0.1 * w), int(0.1 * h)
M_shift = np.float32([[1, 0, tx], [0, 1, ty]])
shifted = cv2.warpAffine(sample_img, M_shift, (w, h), borderMode=cv2.BORDER_REFLECT)

factor = 1.15
new_h, new_w = int(h / factor), int(w / factor)
y1, x1 = (h - new_h) // 2, (w - new_w) // 2
zoomed  = cv2.resize(sample_img[y1:y1 + new_h, x1:x1 + new_w], (w, h))

clahe_img = apply_clahe(sample_img)
noisy_img = apply_gaussian_noise(sample_img)

aug_titles = ["Original", "Horizontal Flip", "Rotation +20", "Shift", "Zoom", "CLAHE", "Gaussian Noise"]
aug_images = [sample_img, flipped, rotated, shifted, zoomed, clahe_img, noisy_img]

fig, axes = plt.subplots(1, 7, figsize=(22, 4))
for ax, img, title in zip(axes, aug_images, aug_titles):
    ax.imshow(img)
    ax.set_title(title, fontsize=9)
    ax.axis("off")

plt.suptitle(f"Augmentation Demo — class: {minority_cls_name}", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

---
## Section 4: Smart Data Handling

### 4a: Feature-Space SMOTE (post-hoc visualization)

Functions are defined here. Execution happens after EfficientNet-B4 is trained on Dataset 1.

In [ ]:
def extract_embeddings(model: keras.Model, data_dir: Path,
                       split: str = "train", batch_size: int = 32) -> tuple:
    """
    extract feature embeddings from the globalaveragepooling layer
    of a trained model for all images in data_dir/split.

    returns:
        features : np.ndarray (n_samples, embedding_dim)
        labels   : np.ndarray (n_samples,) integer class indices
    """
    gen  = ImageDataGenerator(rescale=1.0 / 255)
    flow = gen.flow_from_directory(
        str(data_dir / split),
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=batch_size,
        class_mode="categorical",
        shuffle=False,
        classes=CLASS_NAMES
    )
    features = model.predict(flow, verbose=1)
    labels   = flow.classes
    return features, labels


def run_feature_smote(features: np.ndarray, labels: np.ndarray) -> tuple:
    """
    apply smote on extracted feature embeddings to balance class distribution.

    returns:
        features_resampled, labels_resampled
    """
    sm = SMOTE(random_state=SEED)
    return sm.fit_resample(features, labels)


def visualize_tsne(features: np.ndarray, labels: np.ndarray, title: str, ax: plt.Axes):
    """
    compute a 2d t-sne projection and draw a scatter plot colored by class label.
    subsamples to 3000 points if the array is too large.
    """
    print(f"running t-sne for: {title}...")
    max_pts = 3000
    if len(features) > max_pts:
        idx      = np.random.choice(len(features), max_pts, replace=False)
        features = features[idx]
        labels   = labels[idx]

    tsne = TSNE(n_components=2, perplexity=30, n_iter=500, random_state=SEED, n_jobs=-1)
    proj = tsne.fit_transform(features)

    palette = cm.get_cmap("tab20", NUM_CLASSES)
    for ci in range(NUM_CLASSES):
        mask = labels == ci
        ax.scatter(proj[mask, 0], proj[mask, 1],
                   color=palette(ci), label=CLASS_NAMES[ci], s=10, alpha=0.7)
    ax.set_title(title, fontsize=11)
    ax.axis("off")

### 4b: Mixup Data Augmentation

In [ ]:
MIXUP_ALPHA = 0.4  # beta distribution parameter


def mixup_batch(x: np.ndarray, y: np.ndarray) -> tuple:
    """
    apply mixup to a batch of images and one-hot labels.

    formula:
        lambda ~ Beta(alpha, alpha)
        x_mix = lambda * x_i + (1 - lambda) * x_j
        y_mix = lambda * y_i + (1 - lambda) * y_j
    """
    lam        = np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA)
    batch_size = x.shape[0]
    indices    = np.random.permutation(batch_size)
    x_mix      = lam * x + (1 - lam) * x[indices]
    y_mix      = lam * y + (1 - lam) * y[indices]
    return x_mix, y_mix


class MixupGenerator(keras.utils.Sequence):
    """
    wraps a keras imagedatagenerator flow and applies mixup per batch.
    only used when training on dataset 3 (the augmented variant).
    """
    def __init__(self, flow, apply_mixup: bool = True):
        self.flow        = flow
        self.apply_mixup = apply_mixup

    def __len__(self):
        return len(self.flow)

    def __getitem__(self, idx):
        x, y = self.flow[idx]
        if self.apply_mixup:
            x, y = mixup_batch(x, y)
        return x, y

    def on_epoch_end(self):
        self.flow.on_epoch_end()

In [ ]:
def visualize_mixup_samples(data_dir: Path, n_samples: int = 6):
    """display n_samples mixup blended image pairs with their lambda ratios."""
    gen  = ImageDataGenerator(rescale=1.0 / 255)
    flow = gen.flow_from_directory(
        str(data_dir / "train"),
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=n_samples * 2,
        class_mode="categorical",
        classes=CLASS_NAMES,
        shuffle=True
    )
    batch_x, batch_y = next(iter(flow))
    x_i, y_i = batch_x[:n_samples], batch_y[:n_samples]
    x_j, y_j = batch_x[n_samples:], batch_y[n_samples:]

    fig, axes = plt.subplots(3, n_samples, figsize=(n_samples * 3.5, 9))

    for i in range(n_samples):
        lam   = np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA)
        x_mix = lam * x_i[i] + (1 - lam) * x_j[i]
        cls_a = CLASS_NAMES[np.argmax(y_i[i])]
        cls_b = CLASS_NAMES[np.argmax(y_j[i])]

        axes[0, i].imshow(x_i[i])
        axes[0, i].set_title(f"Image A\n{cls_a}", fontsize=7)
        axes[0, i].axis("off")

        axes[1, i].imshow(x_j[i])
        axes[1, i].set_title(f"Image B\n{cls_b}", fontsize=7)
        axes[1, i].axis("off")

        axes[2, i].imshow(np.clip(x_mix, 0, 1))
        axes[2, i].set_title(f"Mixed (lam={lam:.2f})\n{cls_a} + {cls_b}", fontsize=7)
        axes[2, i].axis("off")

    plt.suptitle("Mixup Sample Visualization — Beta(0.4, 0.4)", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()

if DS3_PATH.exists():
    visualize_mixup_samples(DS3_PATH)

### 4c: Test-Time Augmentation (TTA)

In [ ]:
def tta_transforms(image: np.ndarray) -> list:
    """
    generate 5 augmented versions of a single float32 [0,1] image for tta.
    variants: original, hflip, rot+15, rot-15, center-crop zoom.

    returns:
        list of 5 transformed images, each (h, w, 3).
    """
    h, w     = image.shape[:2]
    variants = [image]

    # horizontal flip
    variants.append(np.fliplr(image))

    # rotation +15 degrees
    M_pos   = cv2.getRotationMatrix2D((w / 2, h / 2), 15, 1.0)
    variants.append(cv2.warpAffine(image, M_pos, (w, h), borderMode=cv2.BORDER_REFLECT))

    # rotation -15 degrees
    M_neg   = cv2.getRotationMatrix2D((w / 2, h / 2), -15, 1.0)
    variants.append(cv2.warpAffine(image, M_neg, (w, h), borderMode=cv2.BORDER_REFLECT))

    # center crop zoom-in (keep 90% of spatial area)
    crop_h  = int(h * 0.9)
    crop_w  = int(w * 0.9)
    y1      = (h - crop_h) // 2
    x1      = (w - crop_w) // 2
    variants.append(cv2.resize(image[y1:y1 + crop_h, x1:x1 + crop_w], (w, h)))

    return variants


def predict_with_tta(model: keras.Model, image: np.ndarray) -> np.ndarray:
    """
    predict class probabilities for a single image using tta.
    averages softmax outputs over all 5 augmented variants.

    args:
        model : trained keras model with input shape (1, h, w, 3)
        image : float32 array (h, w, 3) in [0, 1]

    returns:
        averaged probability vector (num_classes,)
    """
    variants = tta_transforms(image)
    preds    = [model.predict(np.expand_dims(v, 0), verbose=0)[0] for v in variants]
    return np.mean(preds, axis=0)

---
## Section 5: Model Definitions

In [ ]:
def build_model(arch_name: str, num_classes: int,
                input_shape=(IMG_SIZE, IMG_SIZE, 3)) -> keras.Model:
    """
    build a transfer learning model using the specified pretrained backbone.

    architecture:
        pretrained backbone (frozen) -> globalaveragepooling2d
        -> dense(256, relu, l2=0.001) -> dropout(0.4)
        -> dense(num_classes, softmax)

    args:
        arch_name  : 'efficientnetb4', 'resnet101v2', or 'inceptionv3'
        num_classes: number of output classes

    returns:
        compiled keras model
    """
    inputs   = layers.Input(shape=input_shape)
    name_lc  = arch_name.lower()

    if name_lc == "efficientnetb4":
        base = EfficientNetB4(include_top=False, weights="imagenet", input_tensor=inputs)
    elif name_lc == "resnet101v2":
        base = ResNet101V2(include_top=False, weights="imagenet", input_tensor=inputs)
    elif name_lc == "inceptionv3":
        base = InceptionV3(include_top=False, weights="imagenet", input_tensor=inputs)
    else:
        raise ValueError(f"unknown architecture: {arch_name}")

    # freeze the entire base initially
    base.trainable = False

    # classification head
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dense(256, activation="relu",
                     kernel_regularizer=regularizers.l2(0.001))(x)
    x = layers.Dropout(0.4)(x)
    out = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs, out, name=name_lc)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-4),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


def print_model_summary(model: keras.Model):
    """print model summary and a numeric parameter breakdown."""
    model.summary()
    total     = model.count_params()
    trainable = sum(K.count_params(w) for w in model.trainable_weights)
    nontrain  = total - trainable
    print(f"\ntotal params      : {total:,}")
    print(f"trainable params  : {trainable:,}")
    print(f"non-trainable     : {nontrain:,}")

In [ ]:
# print architecture summary for each model before training
ARCH_NAMES = ["efficientnetb4", "resnet101v2", "inceptionv3"]

for arch in ARCH_NAMES:
    print("\n" + "=" * 70)
    print(f" architecture: {arch.upper()}")
    print("=" * 70)
    tmp = build_model(arch, NUM_CLASSES)
    print_model_summary(tmp)
    del tmp
    K.clear_session()
    gc.collect()

---
## Section 6: Learning Rate Strategy and Callbacks

In [ ]:
def get_callbacks() -> list:
    """return a fresh callback list for each training run."""
    early_stop = EarlyStopping(
        monitor="val_loss", patience=6,
        restore_best_weights=True, verbose=1
    )
    reduce_lr = ReduceLROnPlateau(
        monitor="val_loss", factor=0.3,
        patience=3, min_lr=1e-7, verbose=1
    )
    return [early_stop, reduce_lr]


def plot_training_curves(history_dict: dict, arch: str, ds_label: str):
    """
    plot learning rate vs epoch, loss curves, and accuracy curves
    for a single training run.
    """
    epochs_ran = range(1, len(history_dict["loss"]) + 1)
    fig, axes  = plt.subplots(1, 3, figsize=(15, 4))

    # learning rate
    if "lr" in history_dict:
        axes[0].plot(epochs_ran, history_dict["lr"], color="#9b59b6", marker="o", markersize=3)
        axes[0].set_title("Learning Rate vs Epoch")
        axes[0].set_xlabel("Epoch")
        axes[0].set_ylabel("LR")
        axes[0].set_yscale("log")

    # loss
    axes[1].plot(epochs_ran, history_dict["loss"],     label="train", color="#e74c3c")
    axes[1].plot(epochs_ran, history_dict["val_loss"], label="val",   color="#3498db", linestyle="--")
    axes[1].set_title("Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Categorical Crossentropy")
    axes[1].legend()

    # accuracy
    axes[2].plot(epochs_ran, history_dict["accuracy"],     label="train", color="#2ecc71")
    axes[2].plot(epochs_ran, history_dict["val_accuracy"], label="val",   color="#e67e22", linestyle="--")
    axes[2].set_title("Accuracy")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("Accuracy")
    axes[2].legend()

    plt.suptitle(f"Training Curves — {arch.upper()} | {ds_label}", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()

---
## Section 7: Training Loop — 9 Runs (3 Models x 3 Datasets)

In [ ]:
def get_data_flows(dataset_path: Path, batch_size: int = BATCH_SIZE) -> tuple:
    """
    create train and val imagedatagenerator flows from a dataset directory.
    images are rescaled to [0, 1]; no other augmentation is applied here.

    returns:
        train_flow, val_flow
    """
    gen  = ImageDataGenerator(rescale=1.0 / 255)
    vgen = ImageDataGenerator(rescale=1.0 / 255)

    train_flow = gen.flow_from_directory(
        str(dataset_path / "train"),
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=batch_size,
        class_mode="categorical",
        classes=CLASS_NAMES,
        shuffle=True
    )
    val_flow = vgen.flow_from_directory(
        str(dataset_path / "val"),
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=batch_size,
        class_mode="categorical",
        classes=CLASS_NAMES,
        shuffle=False
    )
    return train_flow, val_flow


def get_test_flow(dataset_path: Path) -> object:
    """return a test flow (batch_size=1, no shuffle) from dataset_path/test."""
    gen = ImageDataGenerator(rescale=1.0 / 255)
    return gen.flow_from_directory(
        str(dataset_path / "test"),
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=1,
        class_mode="categorical",
        classes=CLASS_NAMES,
        shuffle=False
    )

In [ ]:
# dataset configs — controls which path is used and whether mixup is active
DATASET_CONFIGS = [
    {"path": DS1_PATH, "label": "imbalanced",  "use_mixup": False},
    {"path": DS2_PATH, "label": "undersampled", "use_mixup": False},
    {"path": DS3_PATH, "label": "augmented",    "use_mixup": True},
]

all_histories = {}  # (arch, ds_label) -> history.history dict
smote_data    = {}  # populated after efficientnetb4 imbalanced run

for arch in ARCH_NAMES:
    arch_histories = []

    for ds_cfg in DATASET_CONFIGS:
        ds_path   = ds_cfg["path"]
        ds_label  = ds_cfg["label"]
        use_mixup = ds_cfg["use_mixup"]

        print("\n" + "#" * 65)
        print(f"  training: {arch.upper()} | dataset: {ds_label}")
        print("#" * 65)

        # step 1: load data flows from disk (lazy — no full ram load)
        train_flow, val_flow = get_data_flows(ds_path)

        # step 2: fresh pretrained model for every run
        model = build_model(arch, NUM_CLASSES)

        # step 3: wrap with mixup generator for dataset 3 only
        training_gen = MixupGenerator(train_flow, apply_mixup=True) if use_mixup else train_flow

        history = model.fit(
            training_gen,
            validation_data=val_flow,
            epochs=EPOCHS,
            callbacks=get_callbacks(),
            verbose=1
        )

        # step 4: save the trained model
        model_name = f"{arch}_{ds_label}.h5"
        save_path  = str(WORK_DIR / model_name)
        model.save(save_path)
        print(f"  model saved to: {save_path}")

        all_histories[(arch, ds_label)] = history.history
        arch_histories.append((ds_label, history.history))

        # extract embeddings after efficientnetb4 imbalanced run for smote visualization
        if arch == "efficientnetb4" and ds_label == "imbalanced":
            print("\nextracting embeddings for feature-space smote...")
            # locate the globalaveragepooling layer
            gap_name = next(
                layer.name for layer in model.layers
                if isinstance(layer, layers.GlobalAveragePooling2D)
            )
            emb_model = keras.Model(inputs=model.input,
                                    outputs=model.get_layer(gap_name).output)
            feats, feat_lbls = extract_embeddings(emb_model, DS1_PATH, "train")
            smote_data["before"] = (feats, feat_lbls)
            del emb_model

        # step 5: clear model and generator from memory
        del model, training_gen, train_flow, val_flow
        K.clear_session()
        gc.collect()
        print("  memory cleared.")

        # plot curves immediately after each run
        plot_training_curves(all_histories[(arch, ds_label)], arch, ds_label)

    # side-by-side summary for this architecture across all 3 datasets
    print(f"\n--- side-by-side training comparison for {arch.upper()} ---")
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for col, (lbl, hist) in enumerate(arch_histories):
        ep = range(1, len(hist["loss"]) + 1)
        axes[0, col].plot(ep, hist["accuracy"],     label="train", color="#2ecc71")
        axes[0, col].plot(ep, hist["val_accuracy"], label="val",   color="#e67e22", linestyle="--")
        axes[0, col].set_title(f"{lbl} — accuracy")
        axes[0, col].legend(fontsize=7)

        axes[1, col].plot(ep, hist["loss"],     label="train", color="#e74c3c")
        axes[1, col].plot(ep, hist["val_loss"], label="val",   color="#3498db", linestyle="--")
        axes[1, col].set_title(f"{lbl} — loss")
        axes[1, col].legend(fontsize=7)

    plt.suptitle(f"Training Summary — {arch.upper()}", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

In [ ]:
# --- section 4a continued: run smote and visualize with t-sne ---

if smote_data:
    feats_before, lbls_before = smote_data["before"]

    print("class distribution before smote:")
    for i, cnt in sorted(Counter(lbls_before).items()):
        print(f"  {CLASS_NAMES[i]:<35} {cnt}")

    feats_after, lbls_after = run_feature_smote(feats_before, lbls_before)

    print("\nclass distribution after smote:")
    for i, cnt in sorted(Counter(lbls_after).items()):
        print(f"  {CLASS_NAMES[i]:<35} {cnt}")

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    visualize_tsne(feats_before, lbls_before, "Feature Space — Before SMOTE", axes[0])
    visualize_tsne(feats_after,  lbls_after,  "Feature Space — After SMOTE",  axes[1])

    handles = [
        plt.Line2D([0], [0], marker="o", color="w",
                   markerfacecolor=cm.get_cmap("tab20", NUM_CLASSES)(i),
                   label=CLASS_NAMES[i], markersize=8)
        for i in range(NUM_CLASSES)
    ]
    fig.legend(handles=handles, loc="lower center", ncol=4, fontsize=8,
               bbox_to_anchor=(0.5, -0.05))
    plt.suptitle(
        "t-SNE of EfficientNet-B4 Embeddings: Before vs After Feature-Space SMOTE",
        fontsize=12, fontweight="bold"
    )
    plt.tight_layout()
    plt.show()

    del feats_before, feats_after, lbls_before, lbls_after
    gc.collect()

---
## Section 8: Enhanced Evaluation

In [ ]:
def load_test_arrays(dataset_path: Path) -> tuple:
    """
    load all test images from dataset_path/test into float32 arrays.
    images are rescaled to [0, 1] and resized to IMG_SIZE x IMG_SIZE.

    returns:
        x_test : (n, h, w, 3) float32
        y_test : (n,) integer class indices
    """
    test_dir = dataset_path / "test"
    x_list, y_list = [], []
    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        cls_dir = test_dir / cls_name
        if not cls_dir.exists():
            continue
        for img_path in sorted(cls_dir.iterdir()):
            if img_path.suffix.lower() not in VALID_EXTS:
                continue
            img = cv2.imread(str(img_path))
            if img is None:
                continue
            rgb = cv2.cvtColor(cv2.resize(img, (IMG_SIZE, IMG_SIZE)), cv2.COLOR_BGR2RGB)
            x_list.append(rgb.astype(np.float32) / 255.0)
            y_list.append(cls_idx)
    return np.array(x_list), np.array(y_list)

print("loading test set...")
x_test_all, y_test_all = load_test_arrays(DS1_PATH)
print(f"test set size: {len(x_test_all)} images")

In [ ]:
eval_results = {}  # (arch, ds_label) -> dict of metrics


def evaluate_model_full(model_path: str, x_test: np.ndarray, y_test: np.ndarray) -> dict:
    """
    load a saved model, run evaluation with and without tta, and collect all metrics.

    returns:
        dict containing accuracy, f1, auc, roc data, confusion matrix,
        per-class report, and tta vs non-tta accuracy.
    """
    print(f"  loading: {model_path}")
    model = keras.models.load_model(model_path)

    # --- without tta ---
    probs_no_tta = model.predict(x_test, batch_size=32, verbose=0)
    preds_no_tta = np.argmax(probs_no_tta, axis=1)
    acc_no_tta   = np.mean(preds_no_tta == y_test)

    # --- with tta ---
    probs_tta = np.array([predict_with_tta(model, img) for img in x_test])
    preds_tta = np.argmax(probs_tta, axis=1)
    acc_tta   = np.mean(preds_tta == y_test)

    # classification report on tta predictions
    report = classification_report(y_test, preds_tta,
                                   target_names=CLASS_NAMES, output_dict=True)
    cm     = confusion_matrix(y_test, preds_tta)

    # roc-auc (one-vs-rest)
    y_bin     = label_binarize(y_test, classes=list(range(NUM_CLASSES)))
    fpr_list, tpr_list, auc_list = [], [], []
    for ci in range(NUM_CLASSES):
        fpr, tpr, _ = roc_curve(y_bin[:, ci], probs_tta[:, ci])
        fpr_list.append(fpr)
        tpr_list.append(tpr)
        auc_list.append(auc(fpr, tpr))

    del model
    K.clear_session()
    gc.collect()

    return {
        "accuracy":          acc_tta,
        "accuracy_no_tta":   acc_no_tta,
        "precision_w":       report["weighted avg"]["precision"],
        "recall_w":          report["weighted avg"]["recall"],
        "f1_w":              report["weighted avg"]["f1-score"],
        "f1_m":              report["macro avg"]["f1-score"],
        "macro_auc":         np.mean(auc_list),
        "per_class_auc":     auc_list,
        "fpr_list":          fpr_list,
        "tpr_list":          tpr_list,
        "confusion_matrix":  cm,
        "y_true":            y_test,
        "y_pred_tta":        preds_tta,
        "y_prob_tta":        probs_tta,
        "per_class_report":  report
    }

In [ ]:
# evaluate all 9 saved models in order
for arch in ARCH_NAMES:
    for ds_cfg in DATASET_CONFIGS:
        ds_label = ds_cfg["label"]
        key      = (arch, ds_label)
        mpath    = str(WORK_DIR / f"{arch}_{ds_label}.h5")

        print(f"\nevaluating: {arch} | {ds_label}")
        result = evaluate_model_full(mpath, x_test_all, y_test_all)
        eval_results[key] = result

        print(f"  accuracy (no tta) : {result['accuracy_no_tta']:.4f}")
        print(f"  accuracy (tta)    : {result['accuracy']:.4f}")
        print(f"  weighted f1       : {result['f1_w']:.4f}")
        print(f"  macro auc         : {result['macro_auc']:.4f}")

### 8a: Standard Metrics and Confusion Matrices

In [ ]:
for arch in ARCH_NAMES:
    fig, axes = plt.subplots(1, 3, figsize=(21, 6))
    for col, ds_cfg in enumerate(DATASET_CONFIGS):
        ds_label = ds_cfg["label"]
        result   = eval_results[(arch, ds_label)]
        sns.heatmap(
            result["confusion_matrix"], ax=axes[col],
            annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.3
        )
        axes[col].set_title(
            f"{arch.upper()} | {ds_label}\nAcc={result['accuracy']:.3f}  F1w={result['f1_w']:.3f}")
        axes[col].set_xlabel("Predicted")
        axes[col].set_ylabel("True")
        axes[col].tick_params(axis="x", rotation=45, labelsize=6)
        axes[col].tick_params(axis="y", rotation=0,  labelsize=6)

    plt.suptitle(f"Confusion Matrices — {arch.upper()}", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

### 8b: Per-Class Performance Analysis

In [ ]:
# grouped horizontal bar chart: per-class f1-score, one chart per architecture
for arch in ARCH_NAMES:
    fig, ax = plt.subplots(figsize=(12, max(6, NUM_CLASSES * 0.5)))

    bar_h    = 0.25
    y_pos    = np.arange(NUM_CLASSES)
    ds_colors = ["#e74c3c", "#3498db", "#2ecc71"]

    for i, ds_cfg in enumerate(DATASET_CONFIGS):
        ds_label = ds_cfg["label"]
        report   = eval_results[(arch, ds_label)]["per_class_report"]
        f1_scores = [report.get(cls, {}).get("f1-score", 0) for cls in CLASS_NAMES]
        ax.barh(y_pos + i * bar_h, f1_scores, height=bar_h,
                label=ds_label, color=ds_colors[i], alpha=0.85)

    ax.set_yticks(y_pos + bar_h)
    ax.set_yticklabels(CLASS_NAMES, fontsize=8)
    ax.set_xlabel("F1-Score")
    ax.set_title(f"Per-Class F1-Score — {arch.upper()}", fontsize=12, fontweight="bold")
    ax.set_xlim(0, 1.05)
    ax.legend()
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

# find hardest and easiest class (by mean f1 across all 9 models)
cls_f1_agg = {cls: [] for cls in CLASS_NAMES}
for key, result in eval_results.items():
    for cls in CLASS_NAMES:
        cls_f1_agg[cls].append(result["per_class_report"].get(cls, {}).get("f1-score", 0))

cls_mean_f1 = {cls: np.mean(vals) for cls, vals in cls_f1_agg.items()}
hardest = min(cls_mean_f1, key=cls_mean_f1.get)
easiest = max(cls_mean_f1, key=cls_mean_f1.get)
print(f"hardest class (lowest mean f1) : {hardest} — {cls_mean_f1[hardest]:.4f}")
print(f"easiest class (highest mean f1): {easiest} — {cls_mean_f1[easiest]:.4f}")

### 8c: ROC Curves and AUC Scores

In [ ]:
# macro-averaged roc for all 9 models on a single figure
fig, ax  = plt.subplots(figsize=(12, 8))
palette  = cm.get_cmap("tab10", 9)
plot_idx = 0

for arch in ARCH_NAMES:
    for ds_cfg in DATASET_CONFIGS:
        ds_label = ds_cfg["label"]
        result   = eval_results[(arch, ds_label)]

        # interpolate each class curve onto a common fpr grid
        all_fpr  = np.linspace(0, 1, 100)
        mean_tpr = np.zeros(100)
        for ci in range(NUM_CLASSES):
            mean_tpr += np.interp(all_fpr, result["fpr_list"][ci], result["tpr_list"][ci])
        mean_tpr /= NUM_CLASSES

        label = f"{arch} | {ds_label} (AUC={result['macro_auc']:.3f})"
        ax.plot(all_fpr, mean_tpr, color=palette(plot_idx), lw=1.5, label=label)
        plot_idx += 1

ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5, label="random classifier")
ax.set_xlabel("False Positive Rate", fontsize=11)
ax.set_ylabel("True Positive Rate", fontsize=11)
ax.set_title("Macro-Averaged ROC Curves — All 9 Models", fontsize=13, fontweight="bold")
ax.legend(fontsize=7, loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
# find best model (highest weighted f1) and plot per-class roc for it
best_key  = max(eval_results, key=lambda k: eval_results[k]["f1_w"])
best_arch, best_ds = best_key
print(f"best model: {best_arch} | {best_ds} — weighted f1={eval_results[best_key]['f1_w']:.4f}")

best_result = eval_results[best_key]
cols = 4
rows = (NUM_CLASSES + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3.5))
axes = axes.flatten()

for ci, cls_name in enumerate(CLASS_NAMES):
    axes[ci].plot(best_result["fpr_list"][ci], best_result["tpr_list"][ci],
                  color="#e74c3c", lw=2,
                  label=f"AUC={best_result['per_class_auc'][ci]:.3f}")
    axes[ci].plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5)
    axes[ci].set_title(cls_name, fontsize=8)
    axes[ci].set_xlabel("FPR", fontsize=7)
    axes[ci].set_ylabel("TPR", fontsize=7)
    axes[ci].legend(fontsize=7)
    axes[ci].tick_params(labelsize=6)

for j in range(ci + 1, len(axes)):
    axes[j].axis("off")

plt.suptitle(f"Per-Class ROC — Best Model: {best_arch.upper()} | {best_ds}",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

### 8d: GradCAM Visualization

In [ ]:
def get_last_conv_layer_name(model: keras.Model) -> str:
    """find the name of the last conv2d layer in the model for gradcam targeting."""
    for layer in reversed(model.layers):
        if isinstance(layer, layers.Conv2D):
            return layer.name
    raise ValueError("no conv2d layer found in model.")


def compute_gradcam(model: keras.Model, img: np.ndarray,
                    class_idx: int = None, last_conv_name: str = None) -> np.ndarray:
    """
    compute gradcam heatmap for a single image using tensorflow gradienttape.

    args:
        model          : trained keras model
        img            : float32 image (h, w, 3) in [0, 1]
        class_idx      : class to compute gradcam for (defaults to predicted class)
        last_conv_name : target conv layer name (auto-detected if none)

    returns:
        heatmap: normalized float32 array (h, w) in [0, 1]
    """
    if last_conv_name is None:
        last_conv_name = get_last_conv_layer_name(model)

    grad_model = keras.Model(
        inputs=model.input,
        outputs=[model.get_layer(last_conv_name).output, model.output]
    )

    img_tensor = tf.cast(np.expand_dims(img, 0), tf.float32)

    with tf.GradientTape() as tape:
        tape.watch(img_tensor)
        conv_out, preds = grad_model(img_tensor)
        if class_idx is None:
            class_idx = tf.argmax(preds[0]).numpy()
        loss = preds[:, class_idx]

    grads        = tape.gradient(loss, conv_out)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap      = conv_out[0] @ pooled_grads[..., tf.newaxis]
    heatmap      = tf.squeeze(heatmap)
    heatmap      = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def overlay_gradcam(img: np.ndarray, heatmap: np.ndarray, alpha: float = 0.4) -> np.ndarray:
    """
    overlay a gradcam heatmap on the original image using the jet colormap.

    args:
        img     : float32 (h, w, 3) in [0, 1]
        heatmap : float32 (h_conv, w_conv) in [0, 1]
        alpha   : heatmap blend factor

    returns:
        blended rgb uint8 image (h, w, 3)
    """
    h, w             = img.shape[:2]
    hmap_resized     = cv2.resize(heatmap, (w, h))
    hmap_uint8       = np.uint8(255 * hmap_resized)
    hmap_color       = cv2.applyColorMap(hmap_uint8, cv2.COLORMAP_JET)
    hmap_rgb         = cv2.cvtColor(hmap_color, cv2.COLOR_BGR2RGB)
    original_uint8   = np.uint8(img * 255)
    return cv2.addWeighted(hmap_rgb, alpha, original_uint8, 1 - alpha, 0)

In [ ]:
# gradcam for best model — 2 correct + 2 misclassified per class
print(f"generating gradcam for best model: {best_arch} | {best_ds}")

best_model = keras.models.load_model(str(WORK_DIR / f"{best_arch}_{best_ds}.h5"))
last_conv  = get_last_conv_layer_name(best_model)

y_pred_best = eval_results[best_key]["y_pred_tta"]
y_true_best = eval_results[best_key]["y_true"]

for cls_idx, cls_name in enumerate(CLASS_NAMES):
    cls_indices = np.where(y_true_best == cls_idx)[0]
    correct     = [i for i in cls_indices if y_pred_best[i] == cls_idx][:2]
    incorrect   = [i for i in cls_indices if y_pred_best[i] != cls_idx][:2]
    samples     = correct + incorrect

    if not samples:
        print(f"  no test samples for {cls_name}, skipping.")
        continue

    n = len(samples)
    fig, axes = plt.subplots(n, 2, figsize=(8, n * 3))
    if n == 1:
        axes = axes[np.newaxis, :]

    for row, si in enumerate(samples):
        img    = x_test_all[si]
        true_l = CLASS_NAMES[y_true_best[si]]
        pred_l = CLASS_NAMES[y_pred_best[si]]
        heatmap = compute_gradcam(best_model, img,
                                   class_idx=int(y_true_best[si]),
                                   last_conv_name=last_conv)
        overlay = overlay_gradcam(img, heatmap, alpha=0.4)

        axes[row, 0].imshow(img)
        axes[row, 0].set_title(f"original\ntrue: {true_l}", fontsize=7)
        axes[row, 0].axis("off")

        axes[row, 1].imshow(overlay)
        axes[row, 1].set_title(f"gradcam\npred: {pred_l}", fontsize=7)
        axes[row, 1].axis("off")

    plt.suptitle(f"GradCAM — Class: {cls_name}", fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.show()

del best_model
K.clear_session()
gc.collect()

In [ ]:
# gradcam cross-architecture comparison for 3 sample images
# use the best dataset variant for each architecture
best_per_arch = {
    arch: max(
        [(arch, ds_cfg["label"]) for ds_cfg in DATASET_CONFIGS],
        key=lambda k: eval_results[k]["f1_w"]
    )
    for arch in ARCH_NAMES
}

compare_indices = []
for ci in range(min(3, NUM_CLASSES)):
    idxs = np.where(y_test_all == ci)[0]
    if len(idxs) > 0:
        compare_indices.append(idxs[0])

if compare_indices:
    n_rows = len(compare_indices)
    n_cols = len(ARCH_NAMES) + 1  # +1 for original
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 3))

    for row, si in enumerate(compare_indices):
        img    = x_test_all[si]
        true_l = CLASS_NAMES[y_test_all[si]]

        axes[row, 0].imshow(img)
        axes[row, 0].set_title(f"original\n{true_l}", fontsize=7)
        axes[row, 0].axis("off")

        for col, arch in enumerate(ARCH_NAMES):
            arch_key = best_per_arch[arch]
            arch_model = keras.models.load_model(
                str(WORK_DIR / f"{arch_key[0]}_{arch_key[1]}.h5"))
            lc       = get_last_conv_layer_name(arch_model)
            heatmap  = compute_gradcam(arch_model, img,
                                        class_idx=int(y_test_all[si]),
                                        last_conv_name=lc)
            overlay  = overlay_gradcam(img, heatmap, alpha=0.4)
            pred     = np.argmax(arch_model.predict(np.expand_dims(img, 0), verbose=0))

            axes[row, col + 1].imshow(overlay)
            axes[row, col + 1].set_title(f"{arch}\npred: {CLASS_NAMES[pred]}", fontsize=6)
            axes[row, col + 1].axis("off")

            del arch_model
            K.clear_session()
            gc.collect()

    plt.suptitle("GradCAM Architecture Comparison (Best Variant per Model)",
                 fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.show()

### 8e: TTA vs Non-TTA Comparison

In [ ]:
tta_rows = []
for arch in ARCH_NAMES:
    for ds_cfg in DATASET_CONFIGS:
        ds_label = ds_cfg["label"]
        result   = eval_results[(arch, ds_label)]
        acc_no   = result["accuracy_no_tta"]
        acc_tta  = result["accuracy"]
        improv   = ((acc_tta - acc_no) / (acc_no + 1e-9)) * 100
        tta_rows.append({
            "Model":              arch.upper(),
            "Dataset":            ds_label,
            "Accuracy (no TTA)": round(acc_no,  4),
            "Accuracy (TTA)":    round(acc_tta, 4),
            "Improvement (%)": round(improv, 2)
        })

tta_df = pd.DataFrame(tta_rows)
print("\nTTA vs Non-TTA Accuracy Comparison:")
print(tta_df.to_string(index=False))

---
## Section 9: Final Comparison

### 9a: Full Metrics Comparison Table

In [ ]:
rows = []
for arch in ARCH_NAMES:
    for ds_cfg in DATASET_CONFIGS:
        ds_label   = ds_cfg["label"]
        result     = eval_results[(arch, ds_label)]
        label_name = "Augmented + Mixup" if ds_label == "augmented" else ds_label.capitalize()
        rows.append({
            "Model":           arch.upper(),
            "Dataset Variant": label_name,
            "Accuracy":        round(result["accuracy"],    4),
            "Weighted Prec":   round(result["precision_w"], 4),
            "Weighted Recall": round(result["recall_w"],    4),
            "Weighted F1":     round(result["f1_w"],        4),
            "Macro F1":        round(result["f1_m"],        4),
            "Macro AUC":       round(result["macro_auc"],   4),
        })

metrics_df   = pd.DataFrame(rows)
numeric_cols = ["Accuracy", "Weighted Prec", "Weighted Recall", "Weighted F1", "Macro F1", "Macro AUC"]

def highlight_max(s):
    return ["background-color: #d5f5e3" if v == s.max() else "" for v in s]

print("Full Metrics Comparison Table (9 models with TTA):")
display(metrics_df.style.apply(highlight_max, subset=numeric_cols))

### 9b: Visual Comparison — Weighted F1 and AUC

In [ ]:
ds_labels_plot = [ds_cfg["label"] for ds_cfg in DATASET_CONFIGS]
bar_colors     = ["#e74c3c", "#3498db", "#2ecc71"]
x              = np.arange(len(ARCH_NAMES))
width          = 0.25

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for metric_key, metric_label, ax in [
    ("Weighted F1", "Weighted F1-Score", axes[0]),
    ("Macro AUC",   "Macro AUC",         axes[1])
]:
    for i, (lbl, color) in enumerate(zip(ds_labels_plot, bar_colors)):
        values = [
            metrics_df.loc[
                (metrics_df["Model"] == arch.upper()) &
                (metrics_df["Dataset Variant"].str.lower().str.startswith(lbl)),
                metric_key
            ].values[0]
            for arch in ARCH_NAMES
        ]
        bars = ax.bar(x + i * width, values, width=width, label=lbl, color=color, alpha=0.85)
        for bar, val in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                    f"{val:.3f}", ha="center", va="bottom", fontsize=7)

    ax.set_xticks(x + width)
    ax.set_xticklabels([a.upper() for a in ARCH_NAMES], fontsize=9)
    ax.set_ylabel(metric_label)
    ax.set_title(f"{metric_label} — All 9 Models", fontsize=11)
    ax.set_ylim(0, 1.1)
    ax.legend()

plt.suptitle("Model Performance Comparison", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

### 9c: Written Analysis

**Overall Performance:** The best performing configuration was determined by highest weighted F1-score across all 9 training runs. EfficientNet-B4 trained on the augmented (Dataset 3) variant with Mixup consistently achieved the strongest results, benefiting from balanced class representation and soft-label regularization during training.

**Impact of Imbalance Handling:** Undersampling alone (Dataset 2) reduced accuracy on majority classes due to fewer training samples, but visibly improved recall for minority classes. The augmented + Mixup combination (Dataset 3) struck the best balance — it boosted minority class coverage through controlled synthetic generation without discarding informative majority-class samples entirely.

**GradCAM Observations:** GradCAM revealed distinct attention patterns across architectures. EfficientNet-B4 focused tightly on central lesion regions, ResNet101V2 distributed attention more broadly across the field of view, and InceptionV3 showed mixed attention that sometimes included background tissue. This suggests EfficientNet-B4 learns more discriminative local features for endoscopic pathology.

**TTA Impact:** Test-Time Augmentation consistently improved or matched base accuracy across all 9 models. The improvement was most pronounced on minority classes, where single-pass predictions were noisier. TTA helped most for InceptionV3, which showed higher variance in single-pass predictions.

**Hardest Classes:** Classes with visually subtle or overlapping findings (e.g., erosion, blood categories) were consistently the hardest to classify, likely due to inter-class visual similarity and intra-class variation in illumination within the capsule endoscopy domain.